# 🎙️ Bengali Speaker Diarization — Option 1: AHC Removed

**Change from base:** `refine_with_ahc()` call completely removed from `diarize_file()`.
BIC is predicting nearly correct speaker counts (9,13,18,13,20 vs GT 11,20,21,10,19).
AHC was collapsing these good estimates down to 3,3,8,3,7. Removing it preserves BIC output.


---
## 📦 Step 0 — Install Dependencies

In [1]:
!pip install speechbrain==0.5.16 --quiet
!pip install silero-vad --quiet
!pip install librosa soundfile --quiet
!pip install scikit-learn scipy --quiet
print('✅ All packages installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.6/630.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 88.4 MB/s eta 0:00:00
✅ All packages installed.


---
## 🔧 Step 1 — Imports & Configuration

In [2]:
import os, json, warnings, math, time, copy, importlib
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import librosa
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.preprocessing import normalize
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


@dataclass
class CFG:
    # ── Paths ─────────────────────────────────────────────────────────────
    train_audio  : Path = Path('/kaggle/input/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train/audio')
    train_annot  : Path = Path('/kaggle/input/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/train/annotation')
    test_audio   : Path = Path('/kaggle/input/dl-sprint-4-0-bengali-speaker-diarization-challenge/diarization/diarization/test/audio')
    working_dir  : Path = Path('/kaggle/working')
    submission   : Path = Path('/kaggle/working/submission.csv')
    ecapa_ckpt   : Path = Path('/kaggle/working/ecapa_arcface.pt')

    # ── Audio ─────────────────────────────────────────────────────────────
    sample_rate  : int   = 16000

    # ── Feature Extraction ────────────────────────────────────────────────
    n_mels     : int   = 80
    n_fft      : int   = 512
    hop_length : int   = 160    # 10 ms
    win_length : int   = 400    # 25 ms
    fmin       : float = 20.0
    fmax       : float = 7600.0

    # ── VAD ───────────────────────────────────────────────────────────────
    vad_threshold      : float = 0.30   # low threshold = more speech detected
    vad_min_speech_ms  : int   = 150
    vad_min_silence_ms : int   = 80

    # ── Sliding Window ────────────────────────────────────────────────────
    window_sec : float = 2.0   # shorter window = more embeddings per file
    step_sec   : float = 0.75  # 0.75s step = overlapping windows
    min_chunk_sec : float = 0.5  # FIX #6: min length to embed tail chunk

    # ── Embeddings ────────────────────────────────────────────────────────
    emb_dim : int = 192

    # ── BIC Speaker Count  ────────────────────────────────────────────────
    # FIX #1: penalty = lam * k * log(n) / 2  [NO 'd' factor]
    max_speakers  : int   = 30
    min_speakers  : int   = 2   # FIX #8: meetings always have >= 2 speakers
    bic_lambda    : float = 1.5  # slightly conservative to avoid over-splitting

    # ── AHC Merge ─────────────────────────────────────────────────────────
    # FIX #5: 0.55 cosine distance = 0.45 cosine similarity (genuine threshold)
    ahc_threshold : float = 0.55

    # ── ArcFace ───────────────────────────────────────────────────────────
    arcface_s   : float = 32.0
    arcface_m   : float = 0.20

    # ── Segment Post-processing ───────────────────────────────────────────
    min_segment_dur : float = 0.3
    merge_gap_sec   : float = 0.15

    # ── Fine-Tuning ───────────────────────────────────────────────────────
    ft_epochs      : int   = 30   # more epochs for better convergence
    ft_lr          : float = 1e-4
    ft_batch_size  : int   = 16
    ft_segment_sec : float = 3.0
    ft_min_seg_sec : float = 0.5


cfg = CFG()
cfg.working_dir.mkdir(parents=True, exist_ok=True)
print(f'\n✅ Config loaded.')

Device: cuda
GPU: Tesla T4

✅ Config loaded.


---
## 🔊 Phase 1 — Preprocessing & Augmentation

In [3]:
def load_audio(path: str, target_sr: int = cfg.sample_rate) -> Tuple[np.ndarray, int]:
    wav, sr = librosa.load(path, sr=target_sr, mono=True)
    return wav.astype(np.float32), sr


def hms_to_sec(t) -> float:
    if isinstance(t, (int, float)):
        return float(t)
    t = str(t).strip()
    parts = t.split(':')
    if len(parts) == 3:
        return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])
    elif len(parts) == 2:
        return int(parts[0]) * 60 + float(parts[1])
    return float(parts[0])


def augment_noise(wav: np.ndarray, snr_db: float = None) -> np.ndarray:
    if snr_db is None:
        snr_db = np.random.uniform(12, 30)
    sig_power   = np.mean(wav ** 2) + 1e-9
    noise_power = sig_power / (10 ** (snr_db / 10))
    noise = np.random.randn(len(wav)).astype(np.float32) * np.sqrt(noise_power)
    return np.clip(wav + noise, -1.0, 1.0)


def augment_reverb(wav: np.ndarray, sr: int = cfg.sample_rate) -> np.ndarray:
    decay    = np.random.uniform(0.05, 0.3)
    rir_len  = int(sr * decay)
    t        = np.arange(rir_len) / sr
    rir      = np.exp(-6.9 * t / decay).astype(np.float32)
    rir     /= (np.sum(rir) + 1e-9)
    return np.clip(np.convolve(wav, rir, mode='full')[:len(wav)], -1.0, 1.0)


def apply_augmentation(wav: np.ndarray, sr: int = cfg.sample_rate, p: float = 0.6) -> np.ndarray:
    if np.random.rand() < p:
        choice = np.random.choice(['noise', 'reverb', 'both'])
        if choice in ('noise', 'both'):
            wav = augment_noise(wav)
        if choice in ('reverb', 'both'):
            wav = augment_reverb(wav, sr)
    wav = wav * np.random.uniform(0.8, 1.0)  # amplitude jitter
    return wav


def extract_log_mel(wav: np.ndarray, sr: int = cfg.sample_rate) -> np.ndarray:
    """Returns [T_frames, 80]"""
    mel = librosa.feature.melspectrogram(
        y=wav, sr=sr,
        n_mels=cfg.n_mels, n_fft=cfg.n_fft,
        hop_length=cfg.hop_length, win_length=cfg.win_length,
        fmin=cfg.fmin, fmax=cfg.fmax, window='hann',
    )
    return librosa.power_to_db(mel, ref=np.max).T.astype(np.float32)


def load_annotation(annot_path: Path) -> List[Dict]:
    df = pd.read_csv(annot_path)
    df.columns = [c.strip().lower() for c in df.columns]
    segments = []
    for _, row in df.iterrows():
        if pd.isna(row['speaker_id']) or pd.isna(row['start_time']) or pd.isna(row['end_time']):
            continue
        start = hms_to_sec(row['start_time'])
        end   = hms_to_sec(row['end_time'])
        if end > start:
            segments.append({'start': start, 'end': end, 'speaker_id': str(int(row['speaker_id']))})
    return segments


def build_training_samples(train_audio_dir: Path, train_annot_dir: Path):
    wav_paths, global_spk, starts, ends = [], [], [], []
    for annot_csv in sorted(train_annot_dir.glob('*.csv')):
        stem     = annot_csv.stem
        wav_path = train_audio_dir / f'{stem}.wav'
        if not wav_path.exists():
            continue
        for seg in load_annotation(annot_csv):
            dur = seg['end'] - seg['start']
            if dur < cfg.ft_min_seg_sec:
                continue
            wav_paths.append(str(wav_path))
            global_spk.append(f"{stem}_SPKR_{seg['speaker_id']}")
            starts.append(seg['start'])
            ends.append(seg['end'])
    unique_spk = sorted(set(global_spk))
    label_map  = {s: i for i, s in enumerate(unique_spk)}
    label_ids  = [label_map[s] for s in global_spk]
    print(f'  {len(wav_paths)} segments | {len(unique_spk)} global speakers')
    return wav_paths, starts, ends, label_ids, label_map


print('✅ Preprocessing functions defined.')

✅ Preprocessing functions defined.


---
## 🔑 Phase 2 — ArcFace Loss & Dataset

In [4]:
class ArcFaceLoss(nn.Module):
    """Additive Angular Margin Loss. s=32, m=0.2 as per paper."""

    def __init__(self, emb_dim: int, n_classes: int,
                 s: float = cfg.arcface_s, m: float = cfg.arcface_m):
        super().__init__()
        self.s = s
        self.m = m
        self.weight    = nn.Parameter(torch.FloatTensor(n_classes, emb_dim))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m     = math.cos(m)
        self.sin_m     = math.sin(m)
        self.th        = math.cos(math.pi - m)
        self.mm        = math.sin(math.pi - m) * m
        self.criterion = nn.CrossEntropyLoss()

    def forward(self, embeddings: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        emb_norm = F.normalize(embeddings, p=2, dim=1)
        w_norm   = F.normalize(self.weight,  p=2, dim=1)
        cosine   = F.linear(emb_norm, w_norm)
        sine     = torch.sqrt((1.0 - cosine.pow(2)).clamp(0, 1))
        phi      = cosine * self.cos_m - sine * self.sin_m
        phi      = torch.where(cosine > self.th, phi, cosine - self.mm)
        one_hot  = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1)
        logits   = (one_hot * phi + (1.0 - one_hot) * cosine) * self.s
        return self.criterion(logits, labels)


class SpeakerSegmentDataset(Dataset):
    def __init__(self, wav_paths, starts, ends, labels,
                 sr=cfg.sample_rate, seg_sec=cfg.ft_segment_sec, augment=True):
        self.wav_paths   = wav_paths
        self.starts      = starts
        self.ends        = ends
        self.labels      = labels
        self.sr          = sr
        self.seg_samples = int(seg_sec * sr)
        self.augment     = augment

    def __len__(self):
        return len(self.wav_paths)

    def __getitem__(self, idx):
        wav, _ = load_audio(self.wav_paths[idx], self.sr)
        s   = int(self.starts[idx] * self.sr)
        e   = int(self.ends[idx]   * self.sr)
        seg = wav[s:e]

        if len(seg) >= self.seg_samples:
            max_start = max(1, len(seg) - self.seg_samples)
            start_i   = np.random.randint(0, max_start) if self.augment else 0
            seg = seg[start_i: start_i + self.seg_samples]
        else:
            seg = np.concatenate([seg, np.zeros(self.seg_samples - len(seg), dtype=np.float32)])

        if self.augment:
            seg = apply_augmentation(seg, self.sr, p=0.6)

        # Ensure correct length
        if len(seg) > self.seg_samples:
            seg = seg[:self.seg_samples]
        elif len(seg) < self.seg_samples:
            seg = np.concatenate([seg, np.zeros(self.seg_samples - len(seg), dtype=np.float32)])

        feats = extract_log_mel(seg, self.sr)  # [T, 80]
        return torch.from_numpy(feats), self.labels[idx]


print('✅ ArcFaceLoss + Dataset defined.')

✅ ArcFaceLoss + Dataset defined.


---
## 🧠 Phase 3 — Load ECAPA-TDNN

In [5]:
os.system('pip install speechbrain==0.5.16 -q')

# Patch fetching.py for newer HuggingFace API
fetching_path = '/usr/local/lib/python3.12/dist-packages/speechbrain/pretrained/fetching.py'
if os.path.exists(fetching_path):
    with open(fetching_path, 'r') as f:
        content = f.read()
    if 'use_auth_token=use_auth_token' in content:
        content = content.replace('use_auth_token=use_auth_token', 'token=use_auth_token')
        with open(fetching_path, 'w') as f:
            f.write(content)
        print('✅ fetching.py patched')

interfaces_path = '/usr/local/lib/python3.12/dist-packages/speechbrain/pretrained/interfaces.py'
if os.path.exists(interfaces_path):
    with open(interfaces_path, 'r') as f:
        content = f.read()
    old = '                filename=pymodule_file,'
    new = '                filename=pymodule_file if pymodule_file is not None else "custom.py",'
    if old in content:
        content = content.replace(old, new)
        with open(interfaces_path, 'w') as f:
            f.write(content)
        print('✅ interfaces.py patched')

savedir = '/kaggle/working/pretrained_ecapa'
os.makedirs(savedir, exist_ok=True)
with open(f'{savedir}/custom.py', 'w') as f:
    f.write('# placeholder\n')

import speechbrain
import speechbrain.pretrained.fetching
import speechbrain.pretrained.interfaces
importlib.reload(speechbrain.pretrained.fetching)
importlib.reload(speechbrain.pretrained.interfaces)
from speechbrain.pretrained.interfaces import SpeakerRecognition


class SpeakerEmbedder:
    """
    FIX #2: embed() calls encoder.forward() DIRECTLY — not pipeline.encode_batch().
    This guarantees fine-tuned weights are actually used at inference.
    FIX #3: After fine-tuning, recompute normalization stats on Bengali training data.
    """

    def __init__(self, savedir: str = '/kaggle/working/pretrained_ecapa'):
        self.pipeline = SpeakerRecognition.from_hparams(
            source='speechbrain/spkrec-ecapa-voxceleb',
            savedir=savedir,
            run_opts={'device': str(device)},
        )
        self.encoder = self.pipeline.mods.embedding_model
        self.encoder.to(device)
        # Normalization stats — will be recomputed after fine-tuning
        self.emb_mean = None
        self.emb_std  = None
        print('✅ ECAPA-TDNN loaded from HuggingFace.')

    def load_finetuned(self, ckpt_path: str):
        """Load fine-tuned weights directly into self.encoder."""
        state = torch.load(ckpt_path, map_location=device)
        self.encoder.load_state_dict(state)
        self.encoder.eval()
        print(f'✅ Fine-tuned encoder loaded: {ckpt_path}')

    def compute_normalization_stats(self, wav_paths: List[str],
                                    starts: List[float], ends: List[float],
                                    n_samples: int = 300, sr: int = cfg.sample_rate):
        """
        FIX #3: Recompute mean/std from Bengali training embeddings
        AFTER fine-tuning so normalization matches the new embedding space.
        """
        print(f'  Computing normalization stats from {n_samples} samples...')
        self.encoder.eval()
        indices = np.random.choice(len(wav_paths), min(n_samples, len(wav_paths)), replace=False)
        embs = []
        with torch.no_grad():
            for i in indices:
                try:
                    wav, _ = load_audio(wav_paths[i], sr)
                    s = int(starts[i] * sr)
                    e = int(ends[i]   * sr)
                    seg = wav[s:e]
                    seg_samples = int(cfg.ft_segment_sec * sr)
                    if len(seg) < int(0.3 * sr):
                        continue
                    if len(seg) > seg_samples:
                        seg = seg[:seg_samples]
                    else:
                        seg = np.concatenate([seg, np.zeros(seg_samples - len(seg), dtype=np.float32)])
                    feats = torch.from_numpy(extract_log_mel(seg, sr)).unsqueeze(0).to(device)  # [1,T,80]
                    lens  = torch.ones(1, device=device)
                    emb   = self.encoder(feats, lens)
                    emb   = emb.squeeze().cpu().numpy()
                    embs.append(emb)
                except Exception:
                    continue
        if embs:
            emb_mat       = np.stack(embs, axis=0)
            self.emb_mean = emb_mat.mean(0)
            self.emb_std  = emb_mat.std(0) + 1e-8
            print(f'  ✅ Normalization stats computed from {len(embs)} embeddings.')
        else:
            print('  ⚠️ Could not compute normalization stats — will use L2 norm only.')

    @torch.no_grad()
    def embed(self, wav: np.ndarray, sr: int = cfg.sample_rate) -> np.ndarray:
        """
        FIX #2: Calls self.encoder DIRECTLY (not pipeline.encode_batch).
        FIX #3: Applies Bengali-recomputed mean/std normalization.
        """
        self.encoder.eval()
        feats = torch.from_numpy(extract_log_mel(wav, sr)).unsqueeze(0).to(device)  # [1,T,80]
        lens  = torch.ones(1, device=device)
        emb   = self.encoder(feats, lens).squeeze().cpu().numpy()  # [192]

        # FIX #3: Apply Bengali normalization if computed
        if self.emb_mean is not None:
            emb = (emb - self.emb_mean) / self.emb_std

        # L2 normalize for cosine similarity clustering
        return emb / (np.linalg.norm(emb) + 1e-8)


embedder = SpeakerEmbedder()

✅ fetching.py patched
✅ interfaces.py patched


hyperparams.yaml: 0.00B [00:00, ?B/s]

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

classifier.ckpt:   0%|          | 0.00/5.53M [00:00<?, ?B/s]

label_encoder.txt: 0.00B [00:00, ?B/s]

✅ ECAPA-TDNN loaded from HuggingFace.


---
## 🏋️ Phase 4 — Fine-Tuning with ArcFace

In [6]:
class FinetuneModel(nn.Module):
    """ECAPA encoder + ArcFace head."""

    def __init__(self, encoder: nn.Module, n_classes: int):
        super().__init__()
        self.encoder = encoder
        self.arcface = ArcFaceLoss(cfg.emb_dim, n_classes, cfg.arcface_s, cfg.arcface_m)

    def forward(self, x: torch.Tensor, labels: torch.Tensor):
        lens = torch.ones(x.size(0), device=x.device)
        emb  = self.encoder(x, lens)
        emb  = emb.squeeze(1) if emb.dim() == 3 else emb  # [B, 192]
        loss = self.arcface(emb, labels)
        return loss, emb


def run_arcface_finetuning(embedder, loader, n_classes, cfg):
    """
    FIX #7: Optimizer created ONCE outside the epoch loop.
    Stage 1 (epochs 1-3): freeze encoder, train ArcFace head only.
    Stage 2 (epochs 4+):  unfreeze all encoder layers, lower LR.
    """
    model = FinetuneModel(embedder.encoder, n_classes).to(device)
    warmup_epochs = 3

    # Stage 1: warm-up ArcFace head with frozen encoder
    for p in model.encoder.parameters():
        p.requires_grad = False
    # FIX #7: optimizer created once per stage, not per epoch
    optimizer_head = torch.optim.AdamW(
        model.arcface.parameters(), lr=cfg.ft_lr * 5, weight_decay=1e-4
    )
    scheduler_head = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_head, T_max=warmup_epochs * len(loader)
    )

    for epoch in range(1, warmup_epochs + 1):
        model.train()
        total_loss, n_batches = 0.0, 0
        for feats, labels in loader:
            feats  = feats.to(device)
            labels = labels.long().to(device)
            optimizer_head.zero_grad()
            loss, _ = model(feats, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.arcface.parameters(), 1.0)
            optimizer_head.step()
            scheduler_head.step()
            total_loss += loss.item()
            n_batches  += 1
        print(f'  Epoch {epoch:2d}/{cfg.ft_epochs} [head-only    ]  loss={total_loss/max(n_batches,1):.4f}')

    # Stage 2: unfreeze all, joint training with lower LR
    for p in model.encoder.parameters():
        p.requires_grad = True
    remaining = cfg.ft_epochs - warmup_epochs
    # FIX #7: ONE optimizer for all remaining epochs
    optimizer_full = torch.optim.AdamW(
        model.parameters(), lr=cfg.ft_lr, weight_decay=1e-4
    )
    scheduler_full = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_full, T_max=remaining * len(loader)
    )

    for epoch in range(warmup_epochs + 1, cfg.ft_epochs + 1):
        model.train()
        total_loss, n_batches = 0.0, 0
        for feats, labels in loader:
            feats  = feats.to(device)
            labels = labels.long().to(device)
            optimizer_full.zero_grad()
            loss, _ = model(feats, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer_full.step()
            scheduler_full.step()
            total_loss += loss.item()
            n_batches  += 1
        print(f'  Epoch {epoch:2d}/{cfg.ft_epochs} [full-encoder  ]  loss={total_loss/max(n_batches,1):.4f}')

    torch.save(model.encoder.state_dict(), str(cfg.ecapa_ckpt))
    print(f'\n✅ Saved: {cfg.ecapa_ckpt}')


# Build training samples
wav_paths, starts, ends, label_ids, label_map = build_training_samples(
    cfg.train_audio, cfg.train_annot
)
n_classes = len(label_map)
print(f'  n_classes = {n_classes}')

ft_dataset = SpeakerSegmentDataset(wav_paths, starts, ends, label_ids, augment=True)
ft_loader  = DataLoader(ft_dataset, batch_size=cfg.ft_batch_size, shuffle=True,
                         num_workers=2, pin_memory=True, drop_last=True)
print(f'✅ Dataset: {len(ft_dataset)} samples | {len(ft_loader)} batches/epoch')

print(f'\n🏋️  ArcFace fine-tuning  epochs={cfg.ft_epochs}  lr={cfg.ft_lr}  s={cfg.arcface_s}  m={cfg.arcface_m}\n')
run_arcface_finetuning(embedder, ft_loader, n_classes, cfg)

  2497 segments | 169 global speakers
  n_classes = 169
✅ Dataset: 2497 samples | 156 batches/epoch

🏋️  ArcFace fine-tuning  epochs=30  lr=0.0001  s=32.0  m=0.2

  Epoch  1/30 [head-only    ]  loss=13.1064
  Epoch  2/30 [head-only    ]  loss=11.7766
  Epoch  3/30 [head-only    ]  loss=11.3244
  Epoch  4/30 [full-encoder  ]  loss=10.2911
  Epoch  5/30 [full-encoder  ]  loss=8.3679
  Epoch  6/30 [full-encoder  ]  loss=7.2995
  Epoch  7/30 [full-encoder  ]  loss=6.3206
  Epoch  8/30 [full-encoder  ]  loss=5.6692
  Epoch  9/30 [full-encoder  ]  loss=4.9864
  Epoch 10/30 [full-encoder  ]  loss=4.5287
  Epoch 11/30 [full-encoder  ]  loss=4.0645
  Epoch 12/30 [full-encoder  ]  loss=3.6526
  Epoch 13/30 [full-encoder  ]  loss=3.2927
  Epoch 14/30 [full-encoder  ]  loss=3.0112
  Epoch 15/30 [full-encoder  ]  loss=2.7425
  Epoch 16/30 [full-encoder  ]  loss=2.5954
  Epoch 17/30 [full-encoder  ]  loss=2.3587
  Epoch 18/30 [full-encoder  ]  loss=2.2373
  Epoch 19/30 [full-encoder  ]  loss=2.1035


In [7]:
# Load fine-tuned weights and recompute normalization stats
embedder.load_finetuned(str(cfg.ecapa_ckpt))

# FIX #3: Recompute normalization stats on Bengali data with the fine-tuned encoder
embedder.compute_normalization_stats(wav_paths, starts, ends, n_samples=400)
print('✅ Fine-tuned encoder + Bengali normalization active.')

✅ Fine-tuned encoder loaded: /kaggle/working/ecapa_arcface.pt
  Computing normalization stats from 400 samples...
  ✅ Normalization stats computed from 400 embeddings.
✅ Fine-tuned encoder + Bengali normalization active.


---
## 📡 Phase 5 — VAD

In [8]:
class NeuralVAD:
    """Silero-VAD wrapper."""

    def __init__(self, threshold: float = cfg.vad_threshold):
        self.threshold = threshold
        self.model, utils = torch.hub.load(
            'snakers4/silero-vad', 'silero_vad', force_reload=False, onnx=False
        )
        self.model.eval().to(device)
        self.get_speech_ts = utils[0]
        print('✅ Silero-VAD loaded.')

    @torch.no_grad()
    def get_speech_segments(self, wav: np.ndarray, sr: int = cfg.sample_rate) -> List[Dict]:
        wav_t = torch.from_numpy(wav).float().to(device)
        return self.get_speech_ts(
            wav_t, self.model,
            threshold=self.threshold,
            sampling_rate=sr,
            min_speech_duration_ms=cfg.vad_min_speech_ms,
            min_silence_duration_ms=cfg.vad_min_silence_ms,
            return_seconds=True,
        )


vad_model = NeuralVAD()

Downloading: "https://github.com/snakers4/silero-vad/zipball/master" to /root/.cache/torch/hub/master.zip
✅ Silero-VAD loaded.


---
## 📊 Phase 6 — Embedding Extraction, BIC & Spectral Clustering

In [9]:
def sliding_window_embeddings(
    wav         : np.ndarray,
    speech_segs : List[Dict],
    emb         : SpeakerEmbedder,
    sr          : int   = cfg.sample_rate,
    window_sec  : float = cfg.window_sec,
    step_sec    : float = cfg.step_sec,
) -> Tuple[List[Tuple[float, float]], np.ndarray]:
    win_samp  = int(window_sec * sr)
    step_samp = int(step_sec   * sr)
    min_samp  = int(cfg.min_chunk_sec * sr)
    windows, emb_list = [], []

    for seg in speech_segs:
        s0  = int(seg['start'] * sr)
        s1  = int(seg['end']   * sr)
        seg_wav = wav[s0:s1]
        if len(seg_wav) < min_samp:
            continue

        if len(seg_wav) <= win_samp:
            # Whole segment is shorter than window: embed directly
            emb_list.append(emb.embed(seg_wav, sr))
            windows.append((seg['start'], seg['end']))
            continue

        ptr = 0
        while ptr + win_samp <= len(seg_wav):
            chunk   = seg_wav[ptr: ptr + win_samp]
            t_start = seg['start'] + ptr / sr
            t_end   = seg['start'] + (ptr + win_samp) / sr
            emb_list.append(emb.embed(chunk, sr))
            windows.append((t_start, t_end))
            ptr += step_samp

        # FIX #6: Embed the trailing partial chunk if long enough
        tail = seg_wav[ptr:]
        if len(tail) >= min_samp:
            t_start = seg['start'] + ptr / sr
            t_end   = seg['end']
            emb_list.append(emb.embed(tail, sr))
            windows.append((t_start, t_end))

    if not emb_list:
        return [], np.empty((0, cfg.emb_dim))
    return windows, np.stack(emb_list, axis=0)


def estimate_n_speakers_bic(
    embeddings   : np.ndarray,
    min_speakers : int   = cfg.min_speakers,
    max_speakers : int   = cfg.max_speakers,
    lam          : float = cfg.bic_lambda,
) -> int:
    """
    FIX #1: CORRECTED BIC — penalty = lam * k * log(n) / 2  (NO 'd' factor).

    The original code used: penalty = lam * k * d * log(n) / 2
    With d=192 (or 448 after fusion) and n~749 windows:
      penalty_at_k2 = 2 * 192 * log(749) / 2 ≈ 2965
      max_log_lik   ≈ 749 (when all cosine sims = 1)
    So the penalty ALWAYS dominated → BIC always returned k=1.

    Fix: remove the 'd' factor so penalty = O(k*log(n)) vs log_lik = O(n).
    At k=2: penalty = 2*log(749)/2 ≈ 6.6  << max_log_lik=749  ✓
    """
    n, d = embeddings.shape
    if n <= 1:
        return max(1, min_speakers)
    if n <= min_speakers:
        return n

    normed  = normalize(embeddings, norm='l2')
    k_max   = min(max_speakers, n - 1)
    k_min   = min(min_speakers, k_max)
    best_k  = k_min
    best_bic = float('inf')

    for k in range(k_min, k_max + 1):
        km     = KMeans(n_clusters=k, n_init=5, random_state=42, max_iter=300)
        labels = km.fit_predict(normed)

        # Log-likelihood: sum of intra-cluster cosine similarities to centroid
        log_lik = 0.0
        for cl in range(k):
            mask = labels == cl
            if mask.sum() < 1:
                continue
            cluster_embs = normed[mask]
            centroid     = cluster_embs.mean(0)
            norm_c       = np.linalg.norm(centroid) + 1e-8
            centroid    /= norm_c
            log_lik     += float(np.sum(cluster_embs @ centroid))

        # FIX #1: penalty WITHOUT 'd' factor
        penalty = lam * k * np.log(n) / 2.0
        bic     = -log_lik + penalty

        if bic < best_bic:
            best_bic = bic
            best_k   = k

    return best_k


def cluster_embeddings_spectral(embeddings: np.ndarray, n_speakers: int) -> np.ndarray:
    """
    Spectral Clustering on cosine affinity matrix.
    Returns cluster labels [N].
    """
    normed = normalize(embeddings, norm='l2')
    n      = len(normed)

    if n_speakers <= 1 or n <= n_speakers:
        return np.zeros(n, dtype=np.int32)

    # Cosine affinity in [0, 1]
    affinity = np.clip((normed @ normed.T + 1.0) / 2.0, 0.0, 1.0)
    try:
        sc = SpectralClustering(
            n_clusters=n_speakers,
            affinity='precomputed',
            n_init=10,
            random_state=42,
            assign_labels='kmeans',
        )
        return sc.fit_predict(affinity).astype(np.int32)
    except Exception:
        # Fallback to KMeans if SpectralClustering fails (e.g. degenerate affinity)
        return KMeans(n_clusters=n_speakers, n_init=10, random_state=42).fit_predict(normed).astype(np.int32)


# OPTION 1: refine_with_ahc REMOVED — AHC was collapsing BIC's correct
# speaker count estimates (9→3, 13→3, 18→8, 13→3, 20→7).
# BIC is already close to ground truth; skipping AHC preserves those estimates.

print('✅ Fixed BIC + Spectral Clustering defined (AHC removed — Option 1).')

✅ Fixed BIC + Spectral Clustering defined (AHC removed — Option 1).


---
## 🔗 Phase 7 — Score-Level Fusion & Segmentation

In [10]:
def score_level_fusion(
    windows    : List[Tuple[float, float]],
    labels     : np.ndarray,
    n_speakers : int,
    frame_step : float = 0.01,
    fill_gaps  : bool  = True,
    max_gap_s  : float = 0.4,
) -> Tuple[np.ndarray, float]:
    if not windows:
        return np.array([], dtype=np.int32), 0.0

    total_dur = max(w[1] for w in windows)
    n_frames  = int(total_dur / frame_step) + 1
    scores    = np.zeros((n_frames, n_speakers), dtype=np.float32)
    counts    = np.zeros(n_frames, dtype=np.float32)

    for (t0, t1), lbl in zip(windows, labels):
        f0 = int(t0 / frame_step)
        f1 = min(int(t1 / frame_step), n_frames)
        if f0 < f1:
            scores[f0:f1, int(lbl)] += 1.0
            counts[f0:f1]           += 1.0

    valid = counts > 0
    scores[valid] /= counts[valid, None]
    frame_labels = np.where(valid, np.argmax(scores, axis=1), -1).astype(np.int32)

    if fill_gaps:
        max_gap_frames = int(max_gap_s / frame_step)
        i = 0
        while i < n_frames:
            if frame_labels[i] == -1:
                j = i
                while j < n_frames and frame_labels[j] == -1:
                    j += 1
                gap_len = j - i
                if gap_len <= max_gap_frames:
                    prev = frame_labels[i - 1] if i > 0 else -1
                    nxt  = frame_labels[j] if j < n_frames else -1
                    fill = prev if prev != -1 else nxt
                    if fill != -1:
                        frame_labels[i:j] = fill
                i = j
            else:
                i += 1

    return frame_labels, total_dur


def frame_labels_to_segments(
    frame_labels : np.ndarray,
    frame_step   : float = 0.01,
    min_dur      : float = cfg.min_segment_dur,
    merge_gap    : float = cfg.merge_gap_sec,
) -> List[Dict]:
    segments = []
    if len(frame_labels) == 0:
        return segments
    cur_lbl   = frame_labels[0]
    start_idx = 0
    for i in range(1, len(frame_labels)):
        if frame_labels[i] != cur_lbl:
            if cur_lbl != -1:
                segments.append({
                    'start_time': round(start_idx * frame_step, 3),
                    'end_time'  : round(i          * frame_step, 3),
                    'speaker_id': f'SPEAKER_{cur_lbl}',
                })
            cur_lbl   = frame_labels[i]
            start_idx = i
    if cur_lbl != -1:
        segments.append({
            'start_time': round(start_idx         * frame_step, 3),
            'end_time'  : round(len(frame_labels) * frame_step, 3),
            'speaker_id': f'SPEAKER_{cur_lbl}',
        })
    segments = [s for s in segments if (s['end_time'] - s['start_time']) >= min_dur]
    merged = []
    for seg in segments:
        if (merged
                and merged[-1]['speaker_id'] == seg['speaker_id']
                and (seg['start_time'] - merged[-1]['end_time']) <= merge_gap):
            merged[-1]['end_time'] = seg['end_time']
        else:
            merged.append(seg)
    return merged


print('✅ Score-level fusion + segmentation defined.')

✅ Score-level fusion + segmentation defined.


---
## 🔗 Phase 8 — Full Inference Pipeline

In [11]:
def diarize_file(
    wav_path  : str,
    vad       : NeuralVAD,
    emb       : SpeakerEmbedder,
    verbose   : bool = False,
) -> List[Dict]:
    """
    Full fixed diarization pipeline:
      1. Load + VAD
      2. Sliding-window ECAPA embeddings (FIX #2: direct encoder, FIX #6: tail chunks)
      3. BIC speaker count (FIX #1: corrected penalty)
      4. Spectral Clustering
      5. AHC refinement REMOVED (Option 1: trust BIC+Spectral directly)
      6. Score-level fusion → segments
    Note: Conformer fusion REMOVED (FIX #4: untrained conformer adds noise)
    """
    wav, sr   = load_audio(wav_path)
    total_dur = len(wav) / sr
    speech    = vad.get_speech_segments(wav, sr)

    if verbose:
        speech_dur = sum(s['end'] - s['start'] for s in speech)
        print(f'  VAD: {len(speech)} segs, {speech_dur:.1f}s / {total_dur:.1f}s speech')

    if not speech:
        return [{'start_time': 0.0, 'end_time': round(total_dur, 3), 'speaker_id': 'SPEAKER_0'}]

    # Phase 2: Extract ECAPA embeddings (FIX #2 + #6 applied inside)
    windows, embeddings = sliding_window_embeddings(wav, speech, emb, sr)
    if verbose:
        print(f'  Embeddings: {len(embeddings)} windows × {embeddings.shape[1]}d')

    if len(embeddings) == 0:
        return [{'start_time': 0.0, 'end_time': round(total_dur, 3), 'speaker_id': 'SPEAKER_0'}]

    # Phase 3: BIC speaker count (FIX #1)
    n_spk = estimate_n_speakers_bic(embeddings)
    if verbose:
        print(f'  BIC → {n_spk} speakers')

    # Phase 4: Spectral Clustering
    labels = cluster_embeddings_spectral(embeddings, n_spk)

    # OPTION 1: AHC step REMOVED — trust BIC+Spectral output directly.
    # AHC at threshold=0.55 was still merging valid speaker clusters
    # because embeddings are not yet perfectly discriminative.
    n_spk = int(labels.max()) + 1
    if verbose:
        counts = {k: int((labels==k).sum()) for k in range(n_spk)}
        print(f'  Spectral result: {n_spk} speakers | sizes: {counts}')

    # Phase 6: Score-level fusion → segments
    frame_labels, _ = score_level_fusion(windows, labels, n_spk)
    segments = frame_labels_to_segments(frame_labels)

    if not segments:
        segments = [{'start_time': 0.0, 'end_time': round(total_dur, 3), 'speaker_id': 'SPEAKER_0'}]

    return segments


print('✅ diarize_file() assembled.')

✅ diarize_file() assembled.


---
## 📈 Phase 9 — DER Evaluation on Training Set

In [12]:
def compute_simple_der(
    ref_segs : List[Dict],
    hyp_segs : List[Dict],
    collar   : float = 0.0,
) -> float:
    if not ref_segs or not hyp_segs:
        return 1.0
    ref_norm = [{'start_time': s['start'], 'end_time': s['end'],
                 'speaker_id': s['speaker_id']} for s in ref_segs]
    frame_step = 0.01
    total_dur  = max(
        max(s['end_time'] for s in ref_norm),
        max(s['end_time'] for s in hyp_segs),
    )
    n_frames = int(total_dur / frame_step) + 1

    def segs_to_mat(segs, spks, ks='start_time', ke='end_time', ksp='speaker_id'):
        mat = np.zeros((n_frames, len(spks)), dtype=np.float32)
        for s in segs:
            f0 = max(0, int((s[ks] + collar) / frame_step))
            f1 = min(n_frames, int((s[ke]   - collar) / frame_step))
            si = spks.index(str(s[ksp]))
            if f0 < f1:
                mat[f0:f1, si] = 1
        return mat

    ref_spks = sorted(set(str(s['speaker_id']) for s in ref_norm))
    hyp_spks = sorted(set(str(s['speaker_id']) for s in hyp_segs))
    ref_mat  = segs_to_mat(ref_norm, ref_spks)
    hyp_mat  = segs_to_mat(hyp_segs, hyp_spks)
    cost     = -(ref_mat.T @ hyp_mat)
    ri, ci   = linear_sum_assignment(cost)
    total_ref = int(ref_mat.sum())
    if total_ref == 0:
        return 0.0
    correct = int(sum(-cost[r, c] for r, c in zip(ri, ci)))
    return float(np.clip(1.0 - correct / total_ref, 0.0, 1.0))


print('📊 DER evaluation on training set (first 5 files)...')
print('-' * 70)

train_files = sorted(cfg.train_audio.glob('*.wav'))[:5]
der_scores  = []

for wav_path in train_files:
    stem     = wav_path.stem
    csv_path = cfg.train_annot / f'{stem}.csv'
    if not csv_path.exists():
        continue

    t0       = time.perf_counter()
    hyp_segs = diarize_file(str(wav_path), vad_model, embedder, verbose=True)
    t1       = time.perf_counter()

    wav_dur  = librosa.get_duration(path=str(wav_path))
    rtf      = (t1 - t0) / wav_dur
    ref_segs = load_annotation(csv_path)
    der      = compute_simple_der(ref_segs, hyp_segs)
    der_scores.append(der)
    n_ref = len(set(s['speaker_id'] for s in ref_segs))
    n_hyp = len(set(s['speaker_id'] for s in hyp_segs))

    print(f'  {stem} | DER={der:.4f} | Score={100*(1-der):.1f} | ref_spk={n_ref} hyp_spk={n_hyp} | RTF={rtf:.3f}')
    print()

if der_scores:
    mean_der   = np.mean(der_scores)
    mean_score = max(0.0, 100.0 * (1.0 - mean_der))
    print(f'  Mean DER   : {mean_der:.4f}')
    print(f'  Mean Score : {mean_score:.2f} / 100')

📊 DER evaluation on training set (first 5 files)...
----------------------------------------------------------------------
  VAD: 792 segs, 1339.0s / 3371.4s speech
  Embeddings: 1251 windows × 192d
  BIC → 12 speakers
  Spectral result: 12 speakers | sizes: {0: 99, 1: 169, 2: 100, 3: 116, 4: 88, 5: 72, 6: 110, 7: 104, 8: 106, 9: 62, 10: 120, 11: 105}
  train_001 | DER=0.7936 | Score=20.6 | ref_spk=11 hyp_spk=12 | RTF=0.025

  VAD: 711 segs, 1579.3s / 2436.5s speech
  Embeddings: 1620 windows × 192d
  BIC → 13 speakers
  Spectral result: 13 speakers | sizes: {0: 88, 1: 193, 2: 158, 3: 173, 4: 92, 5: 74, 6: 159, 7: 93, 8: 121, 9: 120, 10: 213, 11: 58, 12: 78}
  train_002 | DER=0.6916 | Score=30.8 | ref_spk=20 hyp_spk=13 | RTF=0.030

  VAD: 1540 segs, 2844.9s / 3902.0s speech
  Embeddings: 2531 windows × 192d
  BIC → 15 speakers
  Spectral result: 15 speakers | sizes: {0: 212, 1: 209, 2: 198, 3: 251, 4: 115, 5: 157, 6: 148, 7: 70, 8: 188, 9: 176, 10: 98, 11: 179, 12: 151, 13: 231, 14: 14

---
## 🚀 Phase 10 — Inference on Test Set

In [13]:
test_files = sorted(cfg.test_audio.glob('*.wav'))
print(f'Found {len(test_files)} test files.')

results  = []
rtf_list = []

for wav_path in test_files:
    filename = wav_path.stem
    wav_dur  = librosa.get_duration(path=str(wav_path))

    t0 = time.perf_counter()
    try:
        segs = diarize_file(str(wav_path), vad=vad_model, emb=embedder, verbose=False)
    except Exception as ex:
        print(f'  ❌ {filename}: {ex}')
        segs = [{'start_time': 0.0, 'end_time': round(wav_dur, 3), 'speaker_id': 'SPEAKER_0'}]
    t1 = time.perf_counter()

    rtf   = (t1 - t0) / wav_dur
    n_spk = len(set(s['speaker_id'] for s in segs))
    rtf_list.append(rtf)
    print(f'  ✅ {filename} | {len(segs):4d} segs | {n_spk:2d} speakers | {wav_dur:.0f}s | RTF={rtf:.3f}')
    results.append({'filename': filename, 'diarization': json.dumps(segs)})

print(f'\n  Mean RTF : {np.mean(rtf_list):.4f}')

Found 14 test files.
  ✅ test_010 |  927 segs | 14 speakers | 3604s | RTF=0.025
  ✅ test_012 | 1233 segs |  8 speakers | 2504s | RTF=0.030
  ✅ test_016 |  778 segs | 18 speakers | 3411s | RTF=0.026
  ✅ test_018 | 1498 segs | 22 speakers | 3743s | RTF=0.033
  ✅ test_019 | 1138 segs | 21 speakers | 3059s | RTF=0.031
  ✅ test_020 |  857 segs | 15 speakers | 2656s | RTF=0.028
  ✅ test_021 | 1171 segs | 19 speakers | 3099s | RTF=0.032
  ✅ test_022 | 1514 segs | 21 speakers | 3752s | RTF=0.031
  ✅ test_023 | 1282 segs | 18 speakers | 3146s | RTF=0.031
  ✅ test_024 | 1399 segs | 16 speakers | 4008s | RTF=0.030
  ✅ test_027 | 1426 segs | 22 speakers | 4013s | RTF=0.032
  ✅ test_029 |  658 segs | 16 speakers | 2409s | RTF=0.028
  ✅ test_030 |  891 segs | 18 speakers | 3318s | RTF=0.026
  ✅ test_032 |  730 segs | 13 speakers | 2595s | RTF=0.027

  Mean RTF : 0.0293


---
## 📝 Phase 11 — Save Submission

In [14]:
submission_df = pd.DataFrame(results, columns=['filename', 'diarization'])

errors = []
for _, row in submission_df.iterrows():
    try:
        parsed = json.loads(row['diarization'])
        assert isinstance(parsed, list) and len(parsed) > 0
        for seg in parsed:
            assert 'start_time' in seg and 'end_time' in seg and 'speaker_id' in seg
            assert seg['end_time'] > seg['start_time']
    except Exception as ex:
        errors.append(f"{row['filename']}: {ex}")

if errors:
    print('❌ Validation errors:')
    for e in errors:
        print(f'   {e}')
else:
    print('✅ All rows pass validation.')

submission_df.to_csv(cfg.submission, index=False)
print(f'✅ Saved: {cfg.submission}  shape={submission_df.shape}')
print(submission_df.head(3).to_string(max_colwidth=120))

✅ All rows pass validation.
✅ Saved: /kaggle/working/submission.csv  shape=(14, 2)
   filename                                                                                                              diarization
0  test_010  [{"start_time": 13.4, "end_time": 15.4, "speaker_id": "SPEAKER_12"}, {"start_time": 16.39, "end_time": 18.39, "speak...
1  test_012  [{"start_time": 0.9, "end_time": 1.65, "speaker_id": "SPEAKER_4"}, {"start_time": 1.65, "end_time": 3.7, "speaker_id...
2  test_016  [{"start_time": 35.79, "end_time": 36.9, "speaker_id": "SPEAKER_1"}, {"start_time": 39.29, "end_time": 40.9, "speake...


---
## 🔍 Bug Diagnosis Summary

### Why was DER 59% with hyp_spk=1 everywhere?

**Root cause #1 (CRITICAL): BIC penalty had a fatal `d` factor**
```python
# BROKEN (V2):
penalty = lam * k * d * np.log(n) / 2.0
# With d=448 (fused), n=749 windows:
# penalty_at_k=2 = 2 * 448 * log(749) / 2 ≈ 2965
# max_log_lik   = 749  (if all cosine_sims = 1.0)
# penalty > log_lik for ALL k ≥ 2  →  BIC always picks k=1

# FIXED (V3):
penalty = lam * k * np.log(n) / 2.0
# penalty_at_k=2 = 2 * log(749) / 2 ≈ 6.6  <<  749  ✓
```

**Root cause #2 (CRITICAL): Fine-tuned weights not used at inference**
```python
# BROKEN (V2): calls pipeline which has cached OLD encoder weights
emb = self.pipeline.encode_batch(wav_t).squeeze().cpu().numpy()

# FIXED (V3): calls encoder directly
feats = torch.from_numpy(extract_log_mel(wav, sr)).unsqueeze(0).to(device)
emb   = self.encoder(feats, lens).squeeze().cpu().numpy()
```

**Root cause #3 (HIGH): Conformer added noise**
```
Conformer loss: 10.63 → 9.09 (5 epochs, 14% drop)
Well-trained loss target: < 2.0
Concatenating 256d random noise to 192d signal = worse clustering
Fix: Removed conformer fusion from inference. Re-enable only if loss < 2.0.
```

**Root cause #4 (HIGH): VoxCeleb normalization on Bengali encoder**
```python
# BROKEN (V2): mean_var_norm stats from VoxCeleb distort fine-tuned embeddings
# FIXED (V3): recompute mean/std on 400 Bengali training samples after fine-tuning
embedder.compute_normalization_stats(wav_paths, starts, ends, n_samples=400)
```

**Root cause #5 (MEDIUM): AHC threshold inverted semantics**
```python
# BROKEN (V2): threshold=0.15 cosine DISTANCE = 0.85 cosine SIMILARITY
# Same-speaker pairs have sim 0.7–0.9 → almost every pair gets merged

# FIXED (V3): threshold=0.55 cosine DISTANCE = 0.45 cosine SIMILARITY
# Only genuinely identical speakers are merged
```

**Root cause #6 (MEDIUM): Optimizer reset every epoch**
```python
# BROKEN (V2): AdamW recreated each epoch → momentum/velocity reset → slow convergence
for epoch in range(ft_epochs):
    optimizer = torch.optim.AdamW(...)  # ← INSIDE loop!

# FIXED (V3): optimizer created once per training stage, outside loop
optimizer_full = torch.optim.AdamW(...)
for epoch in range(...):
    ...  # optimizer persists across epochs
```

**Root cause #7 (LOW): Sliding window drops tail chunks**
```python
# BROKEN: while ptr + win_samp <= len(seg_wav):
# → final 0–2.9s of each VAD segment silently discarded

# FIXED: after the while loop, embed the tail if >= min_chunk_sec
tail = seg_wav[ptr:]
if len(tail) >= min_samp:
    emb_list.append(emb.embed(tail, sr))
```